# 5. Forecasting and Evaluation

This notebook covers the practical aspects of time series forecasting:
- Train/test split for time series (no shuffling!)
- One-step-ahead (rolling) forecasts
- Forecast error metrics: MAE, RMSE
- Confidence intervals and forecast variance
- Theoretical forecast variance for AR(1)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.arima_process import ArmaProcess
from sklearn.metrics import mean_absolute_error, mean_squared_error

%matplotlib inline
np.random.seed(42)

## 5.1 Simulate ARMA(1,1) Data

We generate synthetic data from an ARMA(1,1) process: $X_t = 0.7 X_{t-1} + \varepsilon_t + 0.3\varepsilon_{t-1}$

In [ ]:
ar = np.array([1, -0.7])
ma = np.array([1, 0.3])
data = ArmaProcess(ar, ma).generate_sample(nsample=300)

train, test = data[:250], data[250:]
print(f"Train: {len(train)} points, Test: {len(test)} points")

## 5.2 Multi-Step Forecast

Fit the model on training data, then forecast all 50 test points at once. The confidence band widens with the forecast horizon.

In [ ]:
model = ARIMA(train, order=(1, 0, 1))
results = model.fit()

forecast = results.get_forecast(steps=50)
pred = forecast.predicted_mean
ci = forecast.conf_int(alpha=0.05)

mae = mean_absolute_error(test, pred)
rmse = np.sqrt(mean_squared_error(test, pred))
print(f"MAE  = {mae:.4f}")
print(f"RMSE = {rmse:.4f}")

## 5.3 Forecast Visualization

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(range(250), train, 'b-', lw=0.7, label='Train')
ax.plot(range(250, 300), test, 'g-', lw=1, label='Test')
ax.plot(range(250, 300), pred, 'r--', lw=1.5, label='Forecast')
ax.fill_between(range(250, 300), ci[:, 0], ci[:, 1],
                alpha=0.2, color='red')
ax.legend()
ax.set_title('ARMA(1,1) Forecast with 95% CI')
plt.tight_layout()
plt.show()

## 5.4 Rolling (One-Step-Ahead) Forecast

A **rolling forecast** re-fits the model at each step using all available data up to that point. It is more realistic but computationally expensive.

In [ ]:
rolling_preds = []
for i in range(len(test)):
    train_i = data[:250+i]
    model_i = ARIMA(train_i, order=(1, 0, 1)).fit()
    rolling_preds.append(model_i.forecast(steps=1)[0])

rmse_multi = np.sqrt(mean_squared_error(test, pred))
rmse_roll = np.sqrt(mean_squared_error(test, rolling_preds))
print(f"RMSE (multi-step):  {rmse_multi:.4f}")
print(f"RMSE (rolling):     {rmse_roll:.4f}")

## 5.5 Theoretical Forecast Variance for AR(1)

For an AR(1) with coefficient $\phi$ and innovation variance $\sigma^2$, the $h$-step forecast variance is:
$$\text{Var}(\hat{X}_{t+h|t}) = \sigma^2 \frac{1 - \phi^{2h}}{1 - \phi^2}$$

As $h \to \infty$, this converges to the unconditional variance $\sigma^2/(1-\phi^2)$.

In [ ]:
phi = 0.8
sigma2 = 1.0
horizons = [1, 2, 5, 10, 20, 50]

print(f"{'h':>5} {'Var(forecast)':>15} {'% of unconditional':>20}")
unc_var = sigma2 / (1 - phi**2)
for h in horizons:
    var_h = sigma2 * (1 - phi**(2*h)) / (1 - phi**2)
    print(f"{h:5d} {var_h:15.4f} {var_h/unc_var*100:19.1f}%")

print(f"\nUnconditional variance: {unc_var:.4f}")

## Key Takeaways

- **Never shuffle** time series data for train/test splits
- **Rolling forecasts** are more accurate but slower than multi-step forecasts
- Forecast **confidence intervals widen** with the horizon
- For AR(1), forecast variance converges to the unconditional variance as $h \to \infty$
- Always report **MAE** and **RMSE** for forecast evaluation